In [1]:
import math
from typing import Iterable, List

import numpy as np
import pandas as pd
import yfinance as yf

In [5]:
def summarize_stocks(tickers: Iterable[str]) -> pd.DataFrame:
    """
    Pull 6 months of daily data for each ticker and compute:
      - price_min: min(Low)
      - price_max: max(High)
      - price_range_abs: max(High) - min(Low)
      - price_range_pct: price_range_abs / avg_close
      - avg_price: average Close over the window
      - hist_vol_annualized: annualized historical vol from daily log returns
      - avg_volume: average daily volume
      - sector
      - industry

    Returns a pandas DataFrame indexed by ticker.
    """
    tickers = [t.strip().upper() for t in tickers if str(t).strip()]
    rows: List[dict] = []

    for ticker in tickers:
        try:
            tk = yf.Ticker(ticker)

            hist = tk.history(period="6mo", interval="1d", auto_adjust=False)

            if hist.empty:
                rows.append(
                    {
                        "ticker": ticker,
                        "sector": None,
                        "industry": None,
                        "start_date": None,
                        "end_date": None,
                        "n_obs": 0,
                        "price_min": np.nan,
                        "price_max": np.nan,
                        "price_range_abs": np.nan,
                        "price_range_pct": np.nan,
                        "avg_price": np.nan,
                        "hist_vol_annualized": np.nan,
                        "avg_volume": np.nan,
                        "avg_volume_m": np.nan,
                        "error": "No price history returned",
                    }
                )
                continue

            # Clean price history
            hist = hist.sort_index().copy()
            hist = hist[["Open", "High", "Low", "Close", "Volume"]].dropna(subset=["Close"])

            # Metrics
            avg_price = hist["Close"].mean()
            min_low = hist["Low"].min()
            max_high = hist["High"].max()
            price_range_abs = max_high - min_low
            price_range_pct = price_range_abs / avg_price if avg_price and not pd.isna(avg_price) else np.nan

            log_returns = np.log(hist["Close"] / hist["Close"].shift(1)).dropna()
            hist_vol_annualized = log_returns.std(ddof=1) * math.sqrt(252) if len(log_returns) > 1 else np.nan

            avg_volume = hist["Volume"].mean()

            # Company metadata
            sector = None
            industry = None
            info_error = None
            try:
                info = tk.info or {}
                sector = info.get("sector")
                industry = info.get("industry")
            except Exception as e:
                info_error = f"info lookup failed: {e}"

            rows.append(
                {
                    "ticker": ticker,
                    "sector": sector,
                    "industry": industry,
                    "start_date": hist.index.min().date(),
                    "end_date": hist.index.max().date(),
                    "n_obs": int(hist["Close"].count()),
                    "price_min": float(min_low),
                    "price_max": float(max_high),
                    "price_range_abs": float(price_range_abs),
                    "price_range_pct": float(price_range_pct) if pd.notna(price_range_pct) else np.nan,
                    "avg_price": float(avg_price),
                    "hist_vol_annualized": float(hist_vol_annualized) if pd.notna(hist_vol_annualized) else np.nan,
                    "avg_volume": float(avg_volume),
                    "avg_volume_m": float(avg_volume / 1e6),
                    "error": info_error,
                }
            )

        except Exception as e:
            rows.append(
                {
                    "ticker": ticker,
                    "sector": None,
                    "industry": None,
                    "start_date": None,
                    "end_date": None,
                    "n_obs": 0,
                    "price_min": np.nan,
                    "price_max": np.nan,
                    "price_range_abs": np.nan,
                    "price_range_pct": np.nan,
                    "avg_price": np.nan,
                    "hist_vol_annualized": np.nan,
                    "avg_volume": np.nan,
                    "avg_volume_m": np.nan,
                    "error": str(e),
                }
            )

    df = pd.DataFrame(rows).set_index("ticker").sort_index()
    return df


In [7]:
tickers = [
    "AAPL",
    "AMZN",
    "NVDA",
    "INTC",
    "TSLA",
    "LCID",
    "COST",
    "RXRX",
    "ISRG",
    "SBUX",
    "CRM",
    "WMT",
    "PEP",
    "CSCO",
    "AMGN",
    "GILD"
]

df = summarize_stocks(tickers)

In [9]:
print("Statistics based on past 6 months of daily data\n")

df[["sector", "industry", "price_min", "price_max", "avg_price", "hist_vol_annualized", "avg_volume_m"]].round(2)

Statistics based on past 6 months of daily data



,sector,industry,price_min,price_max,avg_price,hist_vol_annualized,avg_volume_m
ticker,,,,,,,
AAPL,Technology,Consumer Electronics,243.42,288.62,264.44,0.22,46.47
AMGN,Healthcare,Drug Manufacturers - General,288.00,391.29,337.22,0.28,2.73
AMZN,Consumer Cyclical,Internet Retail,196.00,258.60,224.40,0.32,47.95
COST,Consumer Defensive,Discount Stores,844.06,1028.44,940.94,0.19,2.41
CRM,Technology,Software - Application,174.57,269.11,227.53,0.38,10.50
CSCO,Technology,Communication Equipment,66.81,88.19,76.35,0.31,22.36
GILD,Healthcare,Drug Manufacturers - General,109.66,157.29,131.03,0.27,7.12
INTC,Technology,Semiconductors,32.89,54.60,41.90,0.67,99.02
ISRG,Healthcare,Medical Instruments & Supplies,427.84,603.88,519.75,0.31,1.99
